# RQ2 RP-Geo — frozen near-optimal E2E (T4 x2)

Stage C only. This notebook refuses to train unless a separately created, hash-bound retention artifact is attached. GPU 0 trains RP-Geo; GPU 1 evaluates epoch 50 concurrently once its checkpoint appears, then the two GPUs finish the epoch-100 diagnostics.

## Required inputs

1. CIFAR-100 containing `cifar-100-python/{train,test,meta}`.
2. Gate-A output containing `gate_a_summary.json`.
3. Completed Stage A/B output `rq2-rpgeo-stage-ab-v1.zip`.
4. Separate CPU-freeze output containing `rpgeo_frozen_retention.json`.
5. Kaggle secret `github_token`. Enable T4 x2.

In [ ]:
import os, subprocess, sys, json, time, zipfile, importlib, hashlib, shutil
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Select T4 x2; detected {torch.cuda.device_count()}'
GPU_IDS = (0,1)
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)

In [ ]:
import rq2_e2e_pairwise_pilot as pilot
import scripts.run_e2e_pairwise_pilot as runner
pilot = importlib.reload(pilot); runner = importlib.reload(runner)
INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOT = pilot.find_cifar100_root(INPUT_ROOT)
GATE_A_SUMMARY = pilot.find_gate_a_summary(INPUT_ROOT)
ROOT = pilot.materialize_progress(INPUT_ROOT, '/kaggle/working/e2e_pairwise_pilot_v2', '/kaggle/working/materialized-rpgeo-stage-c')
required = [ROOT/'frozen_protocol.json', ROOT/'resolved_config.yaml', ROOT/'common_warmup/epoch_010.pt', ROOT/'uniform/checkpoints/epoch_100.pt', ROOT/'resource/checkpoints/epoch_100.pt', ROOT/'pure_sw/checkpoints/epoch_100.pt', ROOT/'rpgeo_retention_probe/rpgeo_retention_probe.json', ROOT/'rpgeo_retention_probe/rpgeo_retention_pareto.csv', ROOT/'rpgeo_retention_probe/rpgeo_retention_probe_by_state.csv']
assert all(path.is_file() for path in required), f'Incomplete Stage A/B input: {[str(p) for p in required if not p.is_file()]}'
freeze_candidates = sorted(INPUT_ROOT.rglob('rpgeo_frozen_retention.json'))
by_hash = {}
for path in freeze_candidates: by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), []).append(path)
assert len(by_hash) == 1, f'Expected one content-unique frozen retention artifact, found: {freeze_candidates}'
FREEZE_SOURCE = sorted(next(iter(by_hash.values())), key=lambda p:(len(str(p)),str(p)))[0]
target = ROOT/'rpgeo_frozen_retention.json'
if target.exists(): assert target.read_bytes() == FREEZE_SOURCE.read_bytes(), 'Working retention differs from attached freeze'
else: shutil.copy2(FREEZE_SOURCE, target)
freeze = pilot.load_frozen_rpgeo_retention(ROOT)
assert float(freeze['resource_retention']) == 0.995
assert freeze['selection_source'] == 'mechanistic_pareto_development'
assert freeze['accuracy_used'] is False and freeze['frozen_before_rpgeo_e2e'] is True
print('CIFAR-100:', DATASET_ROOT)
print('Stage A/B root:', ROOT)
print('Frozen retention:', freeze['resource_retention'])

In [ ]:
stage_c_protocol = {
 'status':'FROZEN_RPGEO_STAGE_C', 'seed':3, 'method':'resource_geo',
 'resource_retention':float(freeze['resource_retention']),
 'retention_selection_uses_accuracy':False, 'retention_frozen_before_e2e':True,
 'retention_selection_source':'mechanistic_pareto_development',
 'retention_freeze_sha256':pilot._sha256(ROOT/'rpgeo_frozen_retention.json'),
 'common_epoch10_sha256':pilot._sha256(ROOT/'common_warmup/epoch_010.pt'),
 'gate_a_sha256':pilot._sha256(GATE_A_SUMMARY),
 'validation_used_to_build_policy':False, 'test_used_to_build_policy':False,
 'git_commit':GIT_COMMIT,
}
protocol_path = ROOT/'rpgeo_stage_c_protocol.json'
if protocol_path.exists():
    previous = json.loads(protocol_path.read_text()); keys=[k for k in stage_c_protocol if k!='git_commit']; assert all(previous.get(k)==stage_c_protocol[k] for k in keys)
else: protocol_path.write_text(json.dumps(stage_c_protocol,indent=2)+'\n')
print(json.dumps(stage_c_protocol,indent=2))

## Train only the frozen near-optimal RP-Geo branch

In [ ]:
started = time.perf_counter()
stage_c, branch_runtime, diagnostic_runtime = runner.run_frozen_rpgeo_training(ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS)
decision = pilot.finalize_rpgeo_extension(ROOT)
print(json.dumps(stage_c,indent=2)); print(json.dumps(decision,indent=2))
display(branch_runtime); display(diagnostic_runtime)
refresh = __import__('pandas').read_csv(ROOT/'resource_geo/rpgeo_refresh_metrics.csv')
assert refresh.epoch.tolist() == [10,20,30,40,50,60,70,80,90]
assert (refresh.resource_retention_achieved >= 0.995-1e-8).all()
display(refresh[['epoch','resource_star','resource_rg','resource_retention_achieved','geo_resource','geo_rg','l1_vs_resource']])
display(__import__('pandas').read_csv(ROOT/'rpgeo_frozen_policy_variance_t10_50_100.csv'))
print(f'Elapsed: {(time.perf_counter()-started)/3600:.2f} h')

In [ ]:
required = ['rpgeo_frozen_retention.json','rpgeo_stage_c_protocol.json','resource_geo/checkpoints/epoch_100.pt','resource_geo/training_provenance.json','resource_geo/rpgeo_refresh_metrics.csv','rpgeo_summary.json','rpgeo_method_summary.csv','rpgeo_trajectory_variance_diagnostics.csv','rpgeo_frozen_policy_variance_t10_50_100.csv']
missing = [name for name in required if not (ROOT/name).is_file() or (ROOT/name).stat().st_size == 0]
assert not missing, f'Missing Stage-C artifacts: {missing}'
bundle = Path('/kaggle/working/rq2-rpgeo-frozen-e2e-v1.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in ROOT.rglob('*'):
        if path.is_file(): archive.write(path,Path('e2e_pairwise_pilot_v2')/path.relative_to(ROOT))
print('Persist:',bundle,f'{bundle.stat().st_size/2**30:.2f} GiB')
bundle